1. Зчитування та очищення

(Звантажити та відкрити (вручну або через запропонований скрипт на сайті) наступний датасет: Individual Household Electric Power Consumption Dataset 
Здійснити data cleaning (див. How to Handle Missing Data in Python?))

In [1]:
import pandas as pd
import timeit
from sklearn.preprocessing import StandardScaler, MinMaxScaler

df_power = pd.read_csv('./lab_02/data/household_power_consumption.txt', sep=';', na_values=['?'], low_memory=False)

df_power['Datetime'] = pd.to_datetime(df_power['Date'] + ' ' + df_power['Time'], format='%d/%m/%Y %H:%M:%S')

df_power.dropna(inplace=True)

cols_to_float = ['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2']
df_power[cols_to_float] = df_power[cols_to_float].astype(float)

df_power.head()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00


2. Вибірки та профілювання timeit

(Окремими функціями сформувати вибірки:
Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт.
Обрати всі записи, у яких сила струму лежить в межах 19-20 А, для них виявити ті, у яких пральна машина та холодильних споживають більше, ніж бойлер та кондиціонер.
Обрати випадковим чином 500000 записів (без повторів елементів вибірки), для них обчислити середні величини усіх 3-х груп споживання електричної енергії
Обрати ті записи, які після 18-00 споживають понад 6 кВт за хвилину в середньому, серед відібраних визначити ті, у яких основне споживання електроенергії у вказаний проміжок часу припадає на пральну машину, сушарку, холодильник та освітлення (група 2 є найбільшою), а потім обрати кожен третій результат із першої половини та кожен четвертий результат із другої половини.)


In [2]:
def task_1():
    return df_power[df_power['Global_active_power'] > 5.0]

def task_2():
    return df_power[(df_power['Global_intensity'] >= 19) & 
                    (df_power['Global_intensity'] <= 20) & 
                    (df_power['Sub_metering_2'] > df_power['Sub_metering_3'])]

def task_3():
    sample_df = df_power.sample(n=500000, replace=False)
    return sample_df[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()

def task_4():
    filtered = df_power[(df_power['Datetime'].dt.hour >= 18) & 
                        (df_power['Global_active_power'] > 6)]
    filtered = filtered[(filtered['Sub_metering_2'] > filtered['Sub_metering_1']) & 
                        (filtered['Sub_metering_2'] > filtered['Sub_metering_3'])]
    
    half_idx = len(filtered) // 2
    first_half = filtered.iloc[:half_idx]
    second_half = filtered.iloc[half_idx:]
    
    res1 = first_half.iloc[::3]
    res2 = second_half.iloc[::4]
    
    return pd.concat([res1, res2])

print("Час виконання task_1:", timeit.timeit(task_1, number=10), "секунд (за 10 запусків)")
print("Час виконання task_2:", timeit.timeit(task_2, number=10), "секунд (за 10 запусків)")
print("Час виконання task_4:", timeit.timeit(task_4, number=10), "секунд (за 10 запусків)")

Час виконання task_1: 0.03504539999994449 секунд (за 10 запусків)
Час виконання task_2: 0.08341050000035466 секунд (за 10 запусків)
Час виконання task_4: 0.3845885000000635 секунд (за 10 запусків)


3. Нормалізація, Кореляція та OHE

(Пронормувати та стандартизувати вибраний датасет
Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів.
Провести One Hot Encoding категоріального атрибута.)

In [3]:
scaler_minmax = MinMaxScaler()
scaler_std = StandardScaler()

df_power['GAP_normalized'] = scaler_minmax.fit_transform(df_power[['Global_active_power']])
df_power['GAP_standardized'] = scaler_std.fit_transform(df_power[['Global_active_power']])

pearson_corr = df_power['Global_active_power'].corr(df_power['Global_intensity'], method='pearson')
spearman_corr = df_power['Global_active_power'].corr(df_power['Global_intensity'], method='spearman')

print(f"Пірсон: {pearson_corr:.4f}, Спірмен: {spearman_corr:.4f}")

df_power['Day_of_week'] = df_power['Datetime'].dt.day_name()

df_ohe = pd.get_dummies(df_power['Day_of_week'], prefix='Day')
df_power = pd.concat([df_power, df_ohe], axis=1)

print("Дані після OHE (перші 5 рядків останніх колонок):")
print(df_power.iloc[:5, -7:])

Пірсон: 0.9989, Спірмен: 0.9954
Дані після OHE (перші 5 рядків останніх колонок):
   Day_Friday  Day_Monday  Day_Saturday  Day_Sunday  Day_Thursday  \
0       False       False          True       False         False   
1       False       False          True       False         False   
2       False       False          True       False         False   
3       False       False          True       False         False   
4       False       False          True       False         False   

   Day_Tuesday  Day_Wednesday  
0        False          False  
1        False          False  
2        False          False  
3        False          False  
4        False          False  
